# TFM TUI — Generador de comentarios sintéticos (Madrid)

Genera reseñas sintéticas sobre los recursos **reseñables** (restaurantes, hoteles, cultura, ocio, deporte + POI) a partir de las tablas de **Oro**.

- Volumen configurable (por defecto 40.000), variable por recurso.
- Los **iconos** (Prado, Palacio Real, Thyssen, Retiro…) reciben muchas más reseñas.
- Texto **multilingüe** (ES mayoritario) y **composicional** para dar variedad al topic modeling.
- Rating coherente con el sentimiento del texto (con ~10% de ruido realista).
- Salida con el esquema que espera el notebook de validación.

## 0. Entorno y rutas

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, re, unicodedata
import pandas as pd, numpy as np

BASE = "/content/drive/MyDrive/TFM TUI 3"     # <-- ajusta a tu carpeta
ORO  = f"{BASE}/Oro"
OUT  = f"{BASE}/Comentarios"; os.makedirs(OUT, exist_ok=True)

TOTAL_COMENTARIOS = 40000
rng = np.random.default_rng(42)               # reproducible

## 1. Cargar Oro y construir el universo reseñable

In [ ]:
# 1. UNIVERSO
def construir_universo(poi, rest):
    MACROS_BIZ = {"restaurantes", "hoteles", "cultura", "ocio", "deporte"}
    biz = (rest[rest.macro_categoria.isin(MACROS_BIZ)]
           .drop_duplicates(subset="id_local", keep="first")
           [["id_local", "nombre", "barrio", "macro_categoria", "n_paradas_400m"]]
           .rename(columns={"id_local": "id_origen"}))
    biz["tipo"] = "negocio"

    p = poi[["nombre", "barrio", "macro_categoria", "n_paradas_400m"]].copy()
    p["id_origen"] = ["POI_" + str(i) for i in range(len(p))]
    p["tipo"] = "poi"

    u = pd.concat([biz, p], ignore_index=True)
    u["PK_RELACIONAL"] = ["R" + str(i).zfill(6) for i in range(len(u))]

    # Dominio de reseña (agrupa vocabulario compartido)
    DOMINIO = {"restaurantes": "gastronomia", "hoteles": "alojamiento",
               "cultura": "cultura", "ocio": "ocio", "deporte": "deporte",
               "alimentacion": "mercado", "comercio": "ocio"}
    # POI: refinar por categoria original si es POI
    u["dominio"] = u["macro_categoria"].map(DOMINIO).fillna("ocio")
    return u


poi  = pd.read_parquet(f"{ORO}/POI.parquet")
rest = pd.read_parquet(f"{ORO}/Restaurantes.parquet")

## 2. Popularidad (con boost a los iconos) y reparto de comentarios

In [ ]:
# 2. POPULARIDAD
# Peso base por macro (museos/monumentos/hoteles atraen mas reseñas que un polideportivo)
PESO_BASE = {"cultura": 6.0, "hoteles": 5.0, "restaurantes": 3.0,
             "ocio": 2.5, "alimentacion": 2.5, "comercio": 1.5, "deporte": 1.2}

# Iconos de Madrid: reciben un empujon fuerte (SOLO POI, no negocios con la palabra)
import unicodedata
def _sin_acentos(s):
    return "".join(c for c in unicodedata.normalize("NFD", str(s).upper())
                   if unicodedata.category(c) != "Mn")

ICONOS = (r"MUSEO NACIONAL DEL PRADO|REINA SOFIA|MNCARS|THYSSEN|PALACIO REAL DE MADRID|"
          r"TEMPLO DE DEBOD|MUSEO SOROLLA|ARQUEOLOGICO NACIONAL|DESCALZAS REALES|"
          r"MERCADO DE SAN MIGUEL|REAL JARDIN BOTANICO|BUEN RETIRO|CAIXAFORUM|"
          r"ZOO AQUARIUM|PLAZA MAYOR|PALACIO DE CIBELES|CIBELES|MATADERO|CASA ENCENDIDA|"
          r"MUSEO CERRALBO|LAZARO GALDIANO|MUSEO ABC|CATEDRAL DE LA ALMUDENA|"
          r"ENCARNACION|ERMITA DE SAN ANTONIO|CASON DEL BUEN RETIRO|FAUNIA")

def asignar_popularidad(u):
    base = u["macro_categoria"].map(PESO_BASE).fillna(2.0).values
    # centralidad: paradas de transporte cerca (proxy de afluencia), suave
    par = u["n_paradas_400m"].fillna(0).values
    centralidad = 1.0 + 0.4 * (par / (par.mean() + 1e-9))
    # ruido lognormal: crea la cola larga realista
    ruido = rng.lognormal(mean=0.0, sigma=0.9, size=len(u))
    u["peso"] = base * centralidad * ruido
    # boost SOLO a POI cuyo nombre (sin acentos) casa con un icono
    es_icono = (u["tipo"].eq("poi") &
                u["nombre"].map(_sin_acentos).str.contains(ICONOS, regex=True, na=False))
    u.loc[es_icono, "peso"] *= 30.0
    return u

def repartir_comentarios(u, total=40000):
    p = u["peso"].values / u["peso"].sum()
    u["n_comentarios"] = rng.multinomial(total, p)
    return u



## 3. Ratings realistas (calidad latente por recurso)

In [ ]:
# 3. RATINGS
# Cada recurso tiene una "calidad latente" -> sus reseñas se agrupan (unos gustan, otros no)
def calidad_latente(u):
    # Beta sesgada a positivo (la mayoria de sitios gustan), con variabilidad
    u["calidad"] = rng.beta(5, 2, size=len(u))
    return u

def muestrear_rating(calidad, n):
    """Devuelve n ratings 1-5 en torno a la calidad del sitio."""
    # media de rating entre 2.5 y 5 segun calidad
    media = 2.5 + 2.5 * calidad
    vals = rng.normal(media, 0.9, size=n)
    return np.clip(np.round(vals), 1, 5).astype(int)



## 4. Texto multilingüe composicional

In [ ]:
# 4. TEXTO
IDIOMAS = ["es", "en", "fr", "de", "it", "pt"]
PROB_IDIOMA = [0.62, 0.20, 0.06, 0.05, 0.05, 0.02]

def sent_de_rating(r):
    return "pos" if r >= 4 else ("neg" if r <= 2 else "neu")

# ---- Bancos composicionales: cada reseña = opener + 2-3 fragmentos + closer ----
# Fragmentos ricos en vocabulario del dominio (para que el topic modeling encuentre temas).
ES = {
 "gastronomia": {
  "pos": ["la comida estaba deliciosa","las tapas riquisimas","raciones generosas","productos muy frescos",
          "el pescado y la carne de primera","platos caseros llenos de sabor","el vino de la casa acompaña genial",
          "el servicio atento y cercano","ambiente acogedor","buena relacion calidad-precio","la paella espectacular",
          "postres caseros deliciosos","cocina tradicional bien ejecutada","terraza agradable para cenar"],
  "neu": ["la comida correcta sin destacar","raciones normales","el servicio algo lento pero amable",
          "precio ajustado","ambiente sencillo","carta corta pero suficiente","cumple para una comida rapida"],
  "neg": ["la comida llego fria","tardaron muchisimo en servir","raciones escasas para el precio",
          "caro para lo que ofrecen","el trato fue borde","el local estaba sucio","platos sin sabor",
          "nos cobraron de mas","demasiado ruido dentro"]},
 "alojamiento": {
  "pos": ["habitaciones limpias y comodas","cama muy comoda","ubicacion perfecta y bien comunicado",
          "desayuno variado y rico","personal amabilisimo","excelente descanso","buena relacion calidad-precio",
          "insonorizacion estupenda","recepcion 24 horas muy util"],
  "neu": ["cumple para dormir sin lujos","bien situado aunque paredes finas","habitacion algo pequeña",
          "desayuno mejorable","correcto para una noche"],
  "neg": ["la habitacion no estaba limpia","mucho ruido por la noche","el aire acondicionado no funcionaba",
          "cobros inesperados en recepcion","no coincide con las fotos","wifi pesimo","trato frio del personal"]},
 "cultura": {
  "pos": ["una coleccion impresionante","obras espectaculares","muy bien conservado y explicado",
          "la exposicion temporal merece la pena","audioguia muy completa","espacios amplios y cuidados",
          "una joya del arte en Madrid","perfecto para amantes de la historia","personal informado y amable"],
  "neu": ["interesante aunque se hace corto","habia bastante cola","correcto para una tarde tranquila",
          "merece la pena si pillas entrada gratis","la coleccion permanente esta bien"],
  "neg": ["colas eternas para entrar","precio de entrada elevado","muy masificado","mal señalizado por dentro",
          "esperaba mucho mas","poco accesible","fotos no permitidas y sin explicaciones"]},
 "cultura_mon": {
  "pos": ["un monumento espectacular","precioso sobre todo al atardecer","historia y belleza en cada rincon",
          "impresiona por dentro y por fuera","la fachada es una maravilla","imprescindible para hacer fotos",
          "la arquitectura te deja sin palabras"],
  "neu": ["bonito aunque se visita rapido","esta bien para una foto","hay que ir con tiempo por la gente",
          "el entorno es agradable"],
  "neg": ["demasiada gente para disfrutarlo","descuidado y mal mantenido","no merece lo que cuesta",
          "andamios tapando la fachada","poco que ver por dentro"]},
 "naturaleza": {
  "pos": ["un pulmon verde ideal para pasear","perfecto para desconectar","arboles enormes y mucha sombra",
          "rincones preciosos y tranquilos","genial para ir en familia","ideal para hacer deporte al aire libre",
          "los estanques y jardines muy cuidados","aire puro y ambiente relajado"],
  "neu": ["esta bien aunque los findes se llena","correcto para dar un paseo","le falta mantenimiento en zonas",
          "amplio pero algo descuidado"],
  "neg": ["bastante sucio y descuidado","papeleras desbordadas","poca sombra en verano","ruidoso y masificado",
          "cesped en mal estado"]},
 "deporte": {
  "pos": ["instalaciones limpias y bien equipadas","monitores atentos","perfecto para entrenar",
          "vestuarios amplios y limpios","buena relacion calidad-precio","piscina en buen estado",
          "material moderno y cuidado"],
  "neu": ["cumple aunque a veces esta lleno","correcto para el dia a dia","instalaciones algo antiguas",
          "horarios mejorables"],
  "neg": ["vestuarios sucios","maquinas estropeadas","masificado y mal ventilado","mala atencion",
          "poca limpieza general"]},
 "mercado": {
  "pos": ["productos frescos y mucha variedad","ambiente autentico ideal para picar","trato cercano de los puestos",
          "gran experiencia gastronomica","tapas y vinos estupendos","perfecto para probar de todo"],
  "neu": ["bien aunque algo caro para turistas","correcto para una vuelta rapida","variado pero muy concurrido"],
  "neg": ["precios abusivos","muy masificado","poca variedad real","demasiado turistico, perdio su encanto"]},
 "ocio": {
  "pos": ["un plan estupendo","lo pasamos genial","muy divertido y bien organizado","ideal para ir con amigos",
          "gran ambiente","repetiremos seguro","perfecto para una tarde diferente"],
  "neu": ["esta entretenido sin mas","correcto para pasar el rato","bien aunque un poco caro"],
  "neg": ["aburrido para lo que cuesta","mala organizacion","colas largas","no lo recomendaria"]},
}
ES_OPEN = {"pos": ["","Nos encanto. ","Volveremos seguro. ","Muy recomendable. ","Repetiremos. "],
           "neu": ["","En general, ","La verdad, ","Para ser sinceros, "],
           "neg": ["","Una pena. ","Sinceramente, ","Que decepcion. "]}
ES_CLOSE = {"pos": ["","Un 10.","Lo recomiendo.","Volveremos.","Merece mucho la pena."],
            "neu": ["","Sin mas.","Cumple.","Ni fu ni fa."],
            "neg": ["","No volvere.","No lo recomiendo.","Muy mejorable."]}

EN = {"gen":{
  "pos":["a wonderful experience","the staff were lovely","great value for money","beautiful and well kept",
         "one of the highlights of our trip","clean and welcoming","absolutely worth the visit",
         "we loved every minute","the food was delicious","stunning place"],
  "neu":["it was okay","nothing special but fine","decent for a quick visit","a bit crowded","average overall"],
  "neg":["very disappointing","overpriced for what it is","too crowded to enjoy","poor service",
         "not very clean","would not return","not worth the money"]}}
EN_OPEN={"pos":["","Loved it. ","Highly recommended. "],"neu":["","Overall, ","Honestly, "],"neg":["","Shame. ","Sadly, "]}

MINOR = {
 "fr":{"pos":["une tres belle experience","personnel adorable","excellent rapport qualite-prix","endroit magnifique","on a adore"],
       "neu":["correct sans plus","un peu bonde","bien pour une visite rapide","dans la moyenne"],
       "neg":["tres decevant","trop cher","trop de monde","service mediocre","pas tres propre"]},
 "de":{"pos":["ein wunderbares Erlebnis","freundliches Personal","gutes Preis-Leistungs-Verhaeltnis","toller Ort","hat uns sehr gefallen"],
       "neu":["ganz okay","nichts Besonderes","fuer einen kurzen Besuch in Ordnung","durchschnittlich"],
       "neg":["sehr enttaeuschend","ueberteuert","zu voll","schlechter Service","nicht sauber"]},
 "it":{"pos":["un'esperienza bellissima","personale gentile","ottimo rapporto qualita-prezzo","posto stupendo","ci e piaciuto molto"],
       "neu":["nella media","niente di speciale","va bene per una visita veloce","abbastanza affollato"],
       "neg":["molto deludente","troppo caro","troppa gente","servizio scadente","poco pulito"]},
 "pt":{"pos":["uma experiencia otima","pessoal simpatico","boa relacao qualidade-preco","lugar lindo","adoramos"],
       "neu":["razoavel","nada de especial","bom para uma visita rapida","mediano"],
       "neg":["muito dececionante","caro demais","cheio demais","mau servico","pouco limpo"]},
}
MIN_OPEN={"fr":{"pos":["",""],"neu":[""],"neg":[""]}}  # minoritarios: sin opener extra

def clave_es(dominio, macro, tipo, nombre):
    if dominio == "cultura":
        if tipo == "poi" and re.search(r"MONUMENTO|IGLESIA|BASILICA|CATEDRAL|TEMPLO|PALACIO|PUERTA|PLAZA|FUENTE|ERMITA",
                                       str(nombre).upper()):
            return "cultura_mon"
        return "cultura"
    if macro == "parque" or (dominio == "ocio" and tipo == "poi"):
        return "naturaleza"
    return dominio if dominio in ES else "ocio"

def _componer(fragmentos, opener, closer):
    n = rng.integers(2, 4)                # 2 o 3 fragmentos
    trozos = list(rng.choice(fragmentos, size=min(n, len(fragmentos)), replace=False))
    frase = ", ".join(trozos)
    # capitalizar la primera letra solo si el opener no termina en coma
    if opener == "" or opener.rstrip().endswith("."):
        frase = frase[0].upper() + frase[1:]
    frase += "."
    return (opener + frase + (" " + closer if closer else "")).strip()

def generar_texto(idioma, sent, dom, macro, tipo, nombre):
    if idioma == "es":
        k = clave_es(dom, macro, tipo, nombre)
        return _componer(ES[k][sent], rng.choice(ES_OPEN[sent]), rng.choice(ES_CLOSE[sent]))
    if idioma == "en":
        return _componer(EN["gen"][sent], rng.choice(EN_OPEN[sent]), "")
    return _componer(MINOR[idioma][sent], "", "")



## 5. Autor sintético (sin datos personales reales → RGPD)

In [ ]:
# 5. NOMBRES SINTETICOS (autor)
NOMBRES = ["Ana","Luis","Marta","Carlos","Sofia","Javier","Lucia","Pablo","Elena","Diego",
           "John","Emma","Liam","Olivia","Marie","Hans","Giulia","Marco","Joao","Sophie",
           "Laura","Sergio","Nuria","Ivan","Clara","Tom","Anna","Paul","Chiara","Pedro"]
APELL = ["G.","M.","R.","S.","L.","P.","B.","C.","F.","T.","V.","H."]



## 6. Generar y guardar

In [ ]:
# 6. PIPELINE
def generar(poi, rest, total=40000):
    u = construir_universo(poi, rest)
    u = asignar_popularidad(u)
    u = repartir_comentarios(u, total)
    u = calidad_latente(u)

    filas = []
    fecha_min = pd.Timestamp("2023-01-01"); rango_dias = (pd.Timestamp("2025-12-31") - fecha_min).days
    for row in u.itertuples(index=False):
        n = row.n_comentarios
        if n == 0:
            continue
        ratings = muestrear_rating(row.calidad, n)
        idiomas = rng.choice(IDIOMAS, size=n, p=PROB_IDIOMA)
        dias = rng.integers(0, rango_dias, size=n)
        for i in range(n):
            r = int(ratings[i]); s = sent_de_rating(r)
            # ~10% de ruido realista: el texto salta a un sentimiento ADYACENTE (nunca 5*->negativo total)
            if rng.random() < 0.10:
                s_txt = "neu" if s in ("pos", "neg") else rng.choice(["pos", "neg"])
            else:
                s_txt = s
            texto = generar_texto(idiomas[i], s_txt, row.dominio, row.macro_categoria, row.tipo, row.nombre)
            filas.append((
                row.PK_RELACIONAL, row.nombre, row.barrio, row.macro_categoria,
                texto, r, (fecha_min + pd.Timedelta(days=int(dias[i]))).date(),
                idiomas[i], f"{rng.choice(NOMBRES)} {rng.choice(APELL)}", True))

    df = pd.DataFrame(filas, columns=[
        "PK_RELACIONAL","NOMBRE_RECURSO","BARRIO_RECURSO","CATEGORIA_RECURSO",
        "TEXTO_RESENA","CALIFICACION","FECHA","IDIOMA","AUTOR_SINTETICO","DATASET_SINTETICO"])
    df.insert(0, "ID_RESENA", ["C" + str(i).zfill(7) for i in range(len(df))])
    return df, u


df, u = generar(poi, rest, total=TOTAL_COMENTARIOS)
df.to_csv(f"{OUT}/dataset_completo_tfm.csv", index=False)
u[["PK_RELACIONAL","nombre","barrio","macro_categoria","tipo","n_comentarios"]].to_parquet(
    f"{OUT}/Recursos_resenables.parquet", index=False)
print("Generadas", len(df), "reseñas ->", OUT)
df.head()

## 7. Validación rápida

In [ ]:
print("Nulos:", df.isna().sum().sum(), "| Duplicados:", df.duplicated().sum())
print("Ratings fuera 1-5:", (~df.CALIFICACION.between(1,5)).sum())
print("Textos unicos:", df.TEXTO_RESENA.nunique(), f"({round(100*df.TEXTO_RESENA.nunique()/len(df),1)}%)")
print("Idiomas:", (df.IDIOMA.value_counts(normalize=True)*100).round(1).to_dict())
print("\nTop recursos mas reseñados:")
display(df.groupby("NOMBRE_RECURSO").size().sort_values(ascending=False).head(10))